In [0]:
# Databricks notebook source
# Gold Layer - dim_customer (SCD Type 1)
# Batch MERGE, run before the fact_sales notebook so new/changed customers
# always have a surrogate key available for the fact join.
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
# COMMAND ----------
 
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS retail_sales_dev.gold.dim_customer (
        CustomerSK BIGINT GENERATED ALWAYS AS IDENTITY,
        CustomerID STRING,
        Country STRING,
        CreatedTimestamp TIMESTAMP,
        UpdatedTimestamp TIMESTAMP
    ) USING DELTA
    """
)

In [0]:

silver = spark.table("retail_sales_dev.silver.sales_cleaned")
 
# A CustomerID can appear against more than one Country across invoices in this
# dataset (e.g. address changes, data entry variance). Resolve to the most
# frequent Country per customer rather than the latest, to keep the dimension
# stable against a single noisy row.
country_counts = silver.groupBy("CustomerID", "Country").agg(F.count("*").alias("cnt"))
 
w = Window.partitionBy("CustomerID").orderBy(F.desc("cnt"))

# display(country_counts)

In [0]:

 
customer_source = (
    country_counts
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select("CustomerID", "Country")
    .withColumn("UpdatedTimestamp", F.current_timestamp())
)

customer_source.createOrReplaceTempView("customer_source")
 
# COMMAND ----------
 
spark.sql(
    """
    MERGE INTO retail_sales_dev.gold.dim_customer AS target
    USING customer_source AS source
    ON target.CustomerID = source.CustomerID
    WHEN MATCHED AND target.Country <> source.Country THEN
        UPDATE SET
            target.Country = source.Country,
            target.UpdatedTimestamp = source.UpdatedTimestamp
    WHEN NOT MATCHED THEN
        INSERT (CustomerID, Country, CreatedTimestamp, UpdatedTimestamp)
        VALUES (source.CustomerID, source.Country, current_timestamp(), source.UpdatedTimestamp)
    """
)
 
display(spark.table("retail_sales_dev.gold.dim_customer").orderBy("CustomerSK"))